# 3D Object Classification with ResNet3D-18

In this notebook we classify 3D objects from the **ModelNet10** dataset using a **ResNet3D-18** backbone applied to volumetric (voxelized) representations of 3D meshes.

The notebook is structured as follows:

| Section | Topic |
|---------|-------|
| 0 | Setup & Imports |
| 1 | Data Exploration & Visualization |
| 2 | Dataset & DataLoader |
| 3 | Model Architecture |
| 4 | Training |
| 5 | Evaluation & Prediction Results |

---
## 0. Setup & Imports

In [ ]:
# ============================================================
# All library imports are collected here so you can see
# every dependency at a glance before running any experiment.
# ============================================================
import os
import io
import math
import random
import warnings
import zipfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 — registers 3D projection
from sklearn.metrics import confusion_matrix
from tqdm import tqdm

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import torch
import torch.nn as nn
import torchvision
from torch.utils.data import Dataset, DataLoader

In [ ]:
# ============================================================
# Named constants — change DEMO_MODE to False for full training.
# Using named constants instead of raw numbers ("magic numbers")
# makes the code easier to read and modify from a single place.
# ============================================================

# Set to True  → 2 classes, 16³ voxels, 3 epochs  (< 5 min on CPU)
# Set to False → 10 classes, 32³ voxels, 15 epochs (GPU recommended)
DEMO_MODE = True

VOXEL_RESOLUTION = 16 if DEMO_MODE else 32   # grid side length (D = H = W)
NUM_POINTS       = 3000 if DEMO_MODE else 5000  # surface samples per mesh
BATCH_SIZE       = 8  if DEMO_MODE else 16
NUM_EPOCHS       = 3  if DEMO_MODE else 15
LEARNING_RATE    = 1e-3                         # Adam learning rate
NUM_CLASSES      = 2  if DEMO_MODE else 10
RANDOM_SEED      = 42                           # reproducibility anchor

# Relative paths keep the notebook portable — avoid using absolute paths
# like /home/username/..., which would break on any other machine.
DATASET_PATH    = Path("dataset") / "archive.zip"
EXTRACTED_DIR   = Path("dataset") / "ModelNet10"
CHECKPOINT_PATH = Path("checkpoint_best.pth")

print(f"DEMO_MODE        = {DEMO_MODE}")
print(f"VOXEL_RESOLUTION = {VOXEL_RESOLUTION}")
print(f"NUM_CLASSES      = {NUM_CLASSES}")
print(f"NUM_EPOCHS       = {NUM_EPOCHS}")

In [ ]:
# ============================================================
# Reproducibility — seeds must be set ONCE at the top of the
# notebook before any random operation occurs.
# Setting cudnn flags ensures deterministic GPU convolutions.
# ============================================================
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
# Force deterministic CUDA algorithms (slight speed trade-off for reproducibility)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print(f"All random seeds set to {RANDOM_SEED}")
print("cudnn.deterministic = True  (reproducible GPU results)")


In [ ]:
# ============================================================
# Dataset extraction
# You do not need to unzip anything manually — this cell
# handles extraction on first run and skips it on subsequent runs.
# ============================================================

def extract_dataset(archive_path, destination):
    """
    Extract a zip archive to destination if it has not been extracted yet.

    Args:
        archive_path (Path): Relative path to the .zip file.
        destination  (Path): Directory where contents will be extracted.
    """
    if not archive_path.is_file():
        raise FileNotFoundError(
            f"Dataset archive not found at '{archive_path}'.\n"
            "Make sure you are running the notebook from the intro-3d-cv/ directory "
            "and that dataset/archive.zip is present."
        )
    if destination.is_dir():
        print(f"Dataset already extracted at '{destination}'. Skipping.")
        return
    print(f"Extracting '{archive_path}' → '{destination}' ...")
    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall(destination.parent)
    print("Extraction complete.")


extract_dataset(DATASET_PATH, EXTRACTED_DIR)

---
## 1. Data Exploration & Visualization

### Concept: What is ModelNet10?

**ModelNet10** is a benchmark dataset of 3D CAD models organised into 10 object categories
(bathtub, bed, chair, desk, dresser, monitor, night stand, sofa, table, toilet).
Each model is stored as an **OFF** (Object File Format) file, which encodes a mesh of
triangular faces and their vertices.

### Concept: What is an OFF mesh file?

An OFF file has a simple plain-text structure:
```
OFF
<num_vertices> <num_faces> <num_edges>
x0 y0 z0
x1 y1 z1
...          ← one line per vertex
3 i0 i1 i2  ← each face: vertex count, then vertex indices
...
```

### Concept: Why voxelise?

Convolutional neural networks expect **grid-structured** input (images are 2D grids).
For 3D data we use a **voxel grid**: a fixed-size 3D binary array where each cell is 1
if occupied by surface geometry and 0 otherwise. This converts the irregular mesh into
a format that 3D CNNs can process directly.

In [ ]:
def _read_off_header(file_obj):
    """Read the OFF file header and return (num_verts, num_faces)."""
    first_line = file_obj.readline().strip()
    if first_line == "OFF":
        # Standard variant: counts are on the next line
        counts_line = file_obj.readline().strip()
    elif first_line.startswith("OFF"):
        # Compact variant: counts follow 'OFF' on the same line
        counts_line = first_line[3:].strip()
    else:
        raise ValueError(
            f"Not a valid OFF file — expected header starting with 'OFF', got: {first_line!r}"
        )
    num_verts, num_faces, _ = (int(s) for s in counts_line.split())
    return num_verts, num_faces


def read_off(file_obj):
    """
    Parse an OFF (Object File Format) mesh file.

    Args:
        file_obj: A file-like object opened in text mode.

    Returns:
        vertices (list[list[float]]): N × 3 vertex coordinates.
        faces    (list[list[int]]):   M × k vertex-index lists (k = polygon size).
    """
    num_verts, num_faces = _read_off_header(file_obj)
    vertices = [
        [float(coord) for coord in file_obj.readline().strip().split()]
        for _ in range(num_verts)
    ]
    # Each face line: <polygon_size> <idx0> <idx1> ... — strip the leading count
    faces = [
        [int(idx) for idx in file_obj.readline().strip().split()][1:]
        for _ in range(num_faces)
    ]
    return vertices, faces


#### Expected Output — Dataset Statistics

Running the next cell should print a table like:

```
Class         Train   Test
bathtub         106     50
bed             515    100
chair           889    100
desk            200     86
dresser         200     86
monitor         465    100
night_stand     200     86
sofa            680    100
table           392    100
toilet          344    100
```

In [ ]:
# Scan the extracted directory tree and count samples per class/split
all_classes = sorted(
    entry.name
    for entry in EXTRACTED_DIR.iterdir()
    if entry.is_dir()
)

# Build a class-to-integer label mapping (sorted alphabetically for reproducibility)
CLASS_TO_LABEL = {class_name: idx for idx, class_name in enumerate(all_classes)}
LABEL_TO_CLASS = {idx: class_name for class_name, idx in CLASS_TO_LABEL.items()}

# In demo mode restrict to the first NUM_CLASSES classes
active_classes = all_classes[:NUM_CLASSES]

# Count samples per class per split
print(f"{'Class':<15} {'Train':>6}  {'Test':>6}")
print("-" * 30)
for class_name in all_classes:
    train_count = len(list((EXTRACTED_DIR / class_name / "train").glob("*.off")))
    test_count  = len(list((EXTRACTED_DIR / class_name / "test").glob("*.off")))
    marker = " ◀ active" if class_name in active_classes else ""
    print(f"{class_name:<15} {train_count:>6}  {test_count:>6}{marker}")

print(f"\nActive classes for this run: {active_classes}")

### Concept: Surface Point Sampling

Before voxelising, we sample a fixed number of 3D points **from the mesh surface**.
Random vertex selection would cluster points in regions with many small triangles.
Instead, we use **area-weighted barycentric sampling**: each triangle is selected
proportionally to its area, then a random point inside that triangle is chosen using
barycentric coordinates. This ensures uniform coverage of the surface regardless of
how the mesh is tessellated.

In [ ]:
class PointSampler:
    """
    Sample points uniformly from the surface of a triangular mesh.

    Triangles are selected proportionally to their area so that regions
    with many small triangles are not over-represented.

    Args:
        num_points (int): Number of surface points to sample.
    """

    def __init__(self, num_points):
        assert isinstance(num_points, int) and num_points > 0, (
            f"num_points must be a positive int, got {num_points!r}"
        )
        self.num_points = num_points

    def triangle_area(self, pt1, pt2, pt3):
        """Return the area of the triangle defined by three 3D vertex arrays."""
        side_a = np.linalg.norm(pt1 - pt2)
        side_b = np.linalg.norm(pt2 - pt3)
        side_c = np.linalg.norm(pt3 - pt1)
        semi = 0.5 * (side_a + side_b + side_c)
        return max(semi * (semi - side_a) * (semi - side_b) * (semi - side_c), 0.0) ** 0.5

    def sample_point(self, pt1, pt2, pt3):
        """Sample one random point inside a triangle using barycentric coordinates."""
        # Two uniform draws → barycentric weights via sorted trick
        s, t = sorted([random.random(), random.random()])
        return s * pt1 + (t - s) * pt2 + (1.0 - t) * pt3

    def __call__(self, mesh):
        """
        Sample self.num_points points from the mesh surface.

        Args:
            mesh: tuple (vertices, faces) as returned by read_off.

        Returns:
            np.ndarray of shape (num_points, 3), dtype float32.
        """
        vertices_list, faces_list = mesh
        vertices = np.array(vertices_list, dtype=np.float32)
        areas = np.array([
            self.triangle_area(vertices[f[0]], vertices[f[1]], vertices[f[2]])
            for f in faces_list
        ])
        sampled_faces = random.choices(faces_list, weights=areas.tolist(), k=self.num_points)
        points = np.array([
            self.sample_point(vertices[f[0]], vertices[f[1]], vertices[f[2]])
            for f in sampled_faces
        ], dtype=np.float32)
        return points

In [ ]:
def normalize_point_cloud(points):
    """
    Centre a point cloud at the origin and scale to fit inside a unit sphere.

    Steps:
      1. Subtract the centroid so the cloud is centred at (0, 0, 0).
      2. Divide by the maximum L2 norm so all points lie in [-1, 1]³.

    Args:
        points (np.ndarray): Shape (N, 3).

    Returns:
        np.ndarray of shape (N, 3), same dtype.
    """
    assert points.ndim == 2 and points.shape[1] == 3, (
        f"Expected shape (N, 3), got {points.shape}"
    )
    centred = points - points.mean(axis=0)
    max_norm = np.linalg.norm(centred, axis=1).max()
    if max_norm > 0:
        centred = centred / max_norm
    return centred

### Concept: Voxelisation Algorithm

The `voxelize_point_cloud` function converts a set of 3D surface points into a **binary occupancy grid** — a cube of voxels where each cell is `1.0` if a surface point lands there and `0.0` otherwise.

**Four steps:**
1. **Shift** — subtract the per-axis minimum so the bounding box starts at the origin.
2. **Scale** — divide by the longest-axis extent so all points lie in [0, 1]³.
3. **Quantize** — multiply by `(resolution − 1)` and round to integer grid indices.
4. **Mark** — set `grid[x, y, z] = 1.0` for each occupied voxel.

**Edge cases handled:**
- *Empty cloud* (zero points): return a zero grid and emit a `UserWarning`.
- *Degenerate cloud* (all points identical): step 2 would divide by zero, so we detect `max_extent == 0` and return a zero grid with a `UserWarning` instead of crashing.


In [ ]:
def _normalize_to_unit_cube(points):
    """Shift a point cloud to the origin and scale its longest axis to 1.0.
    Returns None if the cloud is degenerate (all points identical).
    """
    shifted = points - points.min(axis=0)
    max_extent = float(shifted.max())
    if max_extent == 0.0:
        return None
    return shifted / max_extent


def voxelize_point_cloud(points, resolution):
    """Convert an (N, 3) point cloud to a (resolution³) binary voxel grid.
    Degenerate inputs emit a UserWarning and return a zero grid.
    """
    grid = np.zeros((resolution, resolution, resolution), dtype=np.float32)
    if points.shape[0] == 0:
        warnings.warn("voxelize_point_cloud received an empty point cloud; returning zero grid.")
        return grid
    normalized = _normalize_to_unit_cube(points)
    if normalized is None:
        warnings.warn(
            "voxelize_point_cloud received a degenerate point cloud "
            "(all points identical); returning zero grid."
        )
        return grid
    indices = (normalized * (resolution - 1)).astype(np.int32)
    indices = np.clip(indices, 0, resolution - 1)
    grid[indices[:, 0], indices[:, 1], indices[:, 2]] = 1.0
    assert grid.shape == (resolution, resolution, resolution), (
        f"Expected ({resolution},{resolution},{resolution}), got {grid.shape}"
    )
    return grid


#### Expected Output — 3D Point Cloud Visualisation

The next cell renders one interactive 3D scatter plot per active class.
You can **rotate** each plot by clicking and dragging, **zoom** with the scroll
wheel, and **pan** with a right-click drag.

You should see recognisable silhouettes (e.g. a bathtub outline, a bed frame).
Points are coloured by their Z coordinate so depth is perceptible.

In [ ]:
# Load and visualise one sample point cloud per active class (interactive)
sampler_vis = PointSampler(num_points=2000)

num_cols = min(len(active_classes), 4)
num_rows = math.ceil(len(active_classes) / num_cols)

specs = [[{"type": "scatter3d"} for _ in range(num_cols)] for _ in range(num_rows)]
fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    specs=specs,
    subplot_titles=active_classes,
)

for plot_idx, class_name in enumerate(active_classes):
    row = plot_idx // num_cols + 1
    col = plot_idx % num_cols + 1

    sample_files = sorted((EXTRACTED_DIR / class_name / "train").glob("*.off"))
    with open(sample_files[0], "r") as mesh_file:
        vertices, faces = read_off(mesh_file)

    points = sampler_vis((vertices, faces))
    points = normalize_point_cloud(points)

    fig.add_trace(
        go.Scatter3d(
            x=points[:, 0], y=points[:, 1], z=points[:, 2],
            mode="markers",
            marker=dict(size=1.5, color=points[:, 2], colorscale="Viridis", opacity=0.7),
            name=class_name,
            showlegend=False,
        ),
        row=row, col=col,
    )

fig.update_layout(
    title_text="Sample Point Clouds — One Per Class",
    height=400 * num_rows,
)
fig.show()

#### Expected Output — Voxel Grid Visualisation

The next cell voxelises one sample each from the **bathtub** and **bed** classes
and renders them side by side as interactive point clouds of occupied voxel centres.

Rotate each plot by clicking and dragging to compare the two shapes from any angle.
The shape assertion `(VOXEL_RESOLUTION, VOXEL_RESOLUTION, VOXEL_RESOLUTION)` must pass silently.

In [ ]:
# Voxelise one sample each from bathtub and bed
DEMO_CLASSES = ["bathtub", "bed"]
demo_voxel_grids = {}

for class_name in DEMO_CLASSES:
    demo_file = sorted((EXTRACTED_DIR / class_name / "train").glob("*.off"))[0]
    with open(demo_file, "r") as mesh_file:
        demo_verts, demo_faces = read_off(mesh_file)
    demo_points = PointSampler(num_points=NUM_POINTS)((demo_verts, demo_faces))
    demo_points = normalize_point_cloud(demo_points)
    grid = voxelize_point_cloud(demo_points, resolution=VOXEL_RESOLUTION)
    assert grid.shape == (VOXEL_RESOLUTION, VOXEL_RESOLUTION, VOXEL_RESOLUTION), (
        f"{class_name}: expected grid shape "
        f"({VOXEL_RESOLUTION},{VOXEL_RESOLUTION},{VOXEL_RESOLUTION}), got {grid.shape}"
    )
    demo_voxel_grids[class_name] = grid
    print(f"{class_name:<10}: {int(grid.sum()):>4} occupied voxels / {grid.size} total")

# Side-by-side interactive point clouds of the occupied voxel centres
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]],
    subplot_titles=DEMO_CLASSES,
)

for col_idx, class_name in enumerate(DEMO_CLASSES, start=1):
    occ = np.argwhere(demo_voxel_grids[class_name] == 1.0).astype(float)
    fig.add_trace(
        go.Scatter3d(
            x=occ[:, 0], y=occ[:, 1], z=occ[:, 2],
            mode="markers",
            marker=dict(size=2, color=occ[:, 2], colorscale="Viridis", opacity=0.8),
            name=class_name,
            showlegend=False,
        ),
        row=1, col=col_idx,
    )

fig.update_layout(
    title_text=f"Voxelised Point Clouds (resolution = {VOXEL_RESOLUTION}³)",
    height=500,
    width=950,
)
fig.show()

### 🏋️ Exercise — Section 1

Try changing `VOXEL_RESOLUTION` to **64** and re-run Sections 0 and 1.

1. How does the level of geometric detail change in the point cloud and voxel visualisations?
2. What trade-off does a higher resolution introduce for training time and GPU memory?

---
## 2. Dataset & DataLoader

`ModelNet10Dataset` extends PyTorch's `Dataset`: it scans the directory tree,
loads each `.off` file on demand, and runs the full
mesh → point cloud → voxel grid pipeline inside `__getitem__`.

During training, `augment_point_cloud` applies a random Z-axis rotation and small
Gaussian noise to each point cloud before voxelisation, which helps the model
generalise to unseen viewing angles.

In [ ]:
def augment_point_cloud(points):
    """
    Apply random rotation (around Z axis) and Gaussian noise to a point cloud.

    Why Z-axis rotation? ModelNet objects typically sit upright, so rotating around
    the vertical axis simulates viewing the object from different horizontal angles.

    Args:
        points (np.ndarray): Shape (N, 3).

    Returns:
        np.ndarray of shape (N, 3), same dtype.
    """
    # Random rotation around the Z (vertical) axis
    theta = random.random() * 2.0 * math.pi
    cos_t, sin_t = math.cos(theta), math.sin(theta)
    rotation_matrix = np.array(
        [[ cos_t, -sin_t, 0],
         [ sin_t,  cos_t, 0],
         [     0,      0, 1]],
        dtype=np.float32,
    )
    rotated = (rotation_matrix @ points.T).T

    # Small Gaussian noise to simulate measurement imperfection
    noise = np.random.normal(loc=0.0, scale=0.02, size=rotated.shape).astype(np.float32)
    return rotated + noise


class ModelNet10Dataset(Dataset):
    """
    PyTorch Dataset for ModelNet10 voxel-based 3D object classification.

    Each sample is loaded from an .off file, converted to a point cloud,
    optionally augmented, voxelised, and returned as a float32 tensor
    of shape (1, D, H, W) alongside an integer class label.

    Args:
        root_dir        (Path): Path to the extracted ModelNet10/ directory.
        split           (str):  'train' or 'test'.
        active_classes  (list): Class names to include (subset of all 10).
        class_to_label  (dict): Maps class name → integer label.
        voxel_resolution(int):  Voxel grid side length.
        num_points      (int):  Surface sample count per mesh.
        augment         (bool): Apply random rotation + noise (train only).
        max_per_class   (int or None): Cap samples per class (demo mode).
    """

    def __init__(
        self,
        root_dir,
        split,
        active_classes,
        class_to_label,
        voxel_resolution,
        num_points,
        augment=False,
        max_per_class=None,
    ):
        assert split in ("train", "test"), f"split must be 'train' or 'test', got {split!r}"
        self.root_dir         = Path(root_dir)
        self.split            = split
        self.class_to_label   = class_to_label
        self.voxel_resolution = voxel_resolution
        self.num_points       = num_points
        self.augment          = augment
        self.sampler          = PointSampler(num_points)
        self.files            = self._build_file_list(active_classes, max_per_class)

    def _build_file_list(self, active_classes, max_per_class):
        """Scan directory tree and collect {path, label} dicts for all samples."""
        file_list = []
        for class_name in active_classes:
            class_dir = self.root_dir / class_name / self.split
            off_files = sorted(class_dir.glob("*.off"))
            if max_per_class is not None:
                off_files = off_files[:max_per_class]
            for file_path in off_files:
                file_list.append({
                    "path":  file_path,
                    "label": self.class_to_label[class_name],
                })
        return file_list

    def __len__(self):
        return len(self.files)

    def _load_and_voxelize(self, file_path):
        """Load an OFF file and convert it to a voxel tensor."""
        with open(file_path, "r") as mesh_file:
            vertices, faces = read_off(mesh_file)
        points = self.sampler((vertices, faces))
        points = normalize_point_cloud(points)
        if self.augment:
            points = augment_point_cloud(points)
        voxel_grid = voxelize_point_cloud(points, self.voxel_resolution)
        # Add channel dimension: (D, H, W) → (1, D, H, W)
        return torch.from_numpy(voxel_grid).unsqueeze(0)

    def __getitem__(self, idx):
        """
        Return a dict with:
          'voxels' — float32 tensor of shape (1, D, H, W)
          'label'  — integer class label
        """
        sample     = self.files[idx]
        voxel_tensor = self._load_and_voxelize(sample["path"])
        return {"voxels": voxel_tensor, "label": sample["label"]}

#### Expected Output — DataLoader Batch Shape

The assertion below verifies that one batch has shape:
```
voxels: torch.Size([BATCH_SIZE, 1, VOXEL_RESOLUTION, VOXEL_RESOLUTION, VOXEL_RESOLUTION])
labels: torch.Size([BATCH_SIZE])
```
For demo mode (BATCH_SIZE=8, VOXEL_RESOLUTION=16):
```
voxels: torch.Size([8, 1, 16, 16, 16])
labels: torch.Size([8])
```

In [ ]:
# Cap samples per class in demo mode so the DataLoader is quick to construct
max_per_class_train = 50 if DEMO_MODE else None
max_per_class_test  = 20 if DEMO_MODE else None

train_dataset = ModelNet10Dataset(
    root_dir=EXTRACTED_DIR,
    split="train",
    active_classes=active_classes,
    class_to_label=CLASS_TO_LABEL,
    voxel_resolution=VOXEL_RESOLUTION,
    num_points=NUM_POINTS,
    augment=True,
    max_per_class=max_per_class_train,
)

test_dataset = ModelNet10Dataset(
    root_dir=EXTRACTED_DIR,
    split="test",
    active_classes=active_classes,
    class_to_label=CLASS_TO_LABEL,
    voxel_resolution=VOXEL_RESOLUTION,
    num_points=NUM_POINTS,
    augment=False,
    max_per_class=max_per_class_test,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples : {len(train_dataset)}")
print(f"Test  samples : {len(test_dataset)}")
print(f"Train batches : {len(train_loader)}")

# Verify batch shape with an assertion
sample_batch = next(iter(train_loader))
expected_voxel_shape = (BATCH_SIZE, 1, VOXEL_RESOLUTION, VOXEL_RESOLUTION, VOXEL_RESOLUTION)
assert tuple(sample_batch["voxels"].shape) == expected_voxel_shape, (
    f"Expected voxel shape {expected_voxel_shape}, got {tuple(sample_batch['voxels'].shape)}"
)
print(f"\nBatch voxels shape : {sample_batch['voxels'].shape}  ✓")
print(f"Batch labels shape : {sample_batch['label'].shape}")
print(f"Unique labels in batch: {sorted(sample_batch['label'].tolist())}")

### 🏋️ Exercise — Section 2

Add a **random horizontal flip** to `augment_point_cloud`: flip the X coordinates
(`points[:, 0] *= -1`) with probability 0.5.

1. Should this augmentation be applied during training, testing, or both? Why?

---
## 3. Model Architecture

We adapt **torchvision's ResNet3D-18** for single-channel volumetric input.
Two layers are modified from the stock model:
- `stem[0]` — first Conv3d: input channels 3 → 1 (voxel grids are single-channel)
- `fc` — final Linear: output classes 400 → `NUM_CLASSES`

Everything else — the residual blocks, batch norm, pooling — is unchanged.

### ResNet3D-18 Architecture

```
Input  (B, 1, D, H, W)
  │
  ▼
stem   Conv3d(1→64, k=3×7×7, s=1×2×2) + BN + ReLU   →  (B, 64, D, H/2, W/2)
  │
  ▼
layer1  2 × BasicBlock3d(64→64)                        →  (B,  64,   D,   H/2, W/2)
  │
  ▼
layer2  2 × BasicBlock3d(64→128, stride=2)             →  (B, 128, D/2,   H/4, W/4)
  │
  ▼
layer3  2 × BasicBlock3d(128→256, stride=2)            →  (B, 256, D/4,   H/8, W/8)
  │
  ▼
layer4  2 × BasicBlock3d(256→512, stride=2)            →  (B, 512, D/8, H/16, W/16)
  │
  ▼
avgpool AdaptiveAvgPool3d((1,1,1))                      →  (B, 512, 1, 1, 1)
  │
  ▼
flatten                                                 →  (B, 512)
  │
  ▼
fc      Linear(512 → num_classes)                      →  (B, num_classes)
```

In [ ]:
def _adapt_stem_conv(model, in_channels):
    """Replace stem[0] to accept in_channels input channels instead of 3."""
    original_conv = model.stem[0]
    model.stem[0] = nn.Conv3d(
        in_channels=in_channels,
        out_channels=original_conv.out_channels,
        kernel_size=original_conv.kernel_size,
        stride=original_conv.stride,
        padding=original_conv.padding,
        bias=False,
    )


def build_resnet3d18_classifier(num_classes, in_channels=1):
    """
    Adapt torchvision's ResNet3D-18 for single-channel volumetric classification.

    Only two layers are modified from the standard model:
      1. stem[0] — first Conv3d: input channels 3 → in_channels.
      2. fc      — final Linear: output classes 400 → num_classes.

    Args:
        num_classes (int): Number of output classes.
        in_channels (int): Input channel count (default 1 for voxel grids).

    Returns:
        torch.nn.Module: The adapted ResNet3D-18.
    """
    model = torchvision.models.video.r3d_18(weights=None)
    _adapt_stem_conv(model, in_channels)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model


#### Expected Output — Model Summary

```
All random seeds set to 42
cudnn.deterministic = True  (reproducible GPU results)
Using device: cuda  (or cpu)

Total trainable parameters: 33,148,482   (demo 2-class model)
                             33,152,586   (full 10-class model)
Forward pass output shape: torch.Size([1, 2])  ✓
```


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cpu":
    print(
        "⚠️  No GPU detected. Training will run on CPU.\n"
        f"   Demo mode ({NUM_EPOCHS} epochs, {VOXEL_RESOLUTION}³ voxels) "
        "should complete in under 5 minutes."
    )

model = build_resnet3d18_classifier(num_classes=NUM_CLASSES, in_channels=1)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params:,}")

# Verify forward-pass output shape with a dummy batch
with torch.no_grad():
    dummy_input = torch.zeros(1, 1, VOXEL_RESOLUTION, VOXEL_RESOLUTION, VOXEL_RESOLUTION,
                              device=device)
    dummy_output = model(dummy_input)

assert dummy_output.shape == (1, NUM_CLASSES), (
    f"Expected output shape (1, {NUM_CLASSES}), got {dummy_output.shape}"
)
print(f"Forward pass output shape: {dummy_output.shape}  ✓")

### 🏋️ Exercise — Section 3

Freeze the backbone and train only the classifier head:

```python
for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True
```

1. How many trainable parameters remain? How does convergence speed compare to the full model?

---
## 4. Training

We use **cross-entropy loss** and the **Adam optimiser** (`lr=LEARNING_RATE`).
The model is saved to `CHECKPOINT_PATH` whenever validation accuracy improves,
so evaluation always uses the best weights rather than the last epoch's.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """
    Run one full pass over the training DataLoader and update model weights.

    Args:
        model     : The neural network.
        loader    : Training DataLoader.
        optimizer : Optimiser (e.g. Adam).
        criterion : Loss function (e.g. CrossEntropyLoss).
        device    : torch.device to run on.

    Returns:
        float: Mean loss over all batches in this epoch.
    """
    model.train()
    total_loss = 0.0

    for batch in tqdm(loader, desc="  Training", leave=False):
        voxels = batch["voxels"].to(device)          # (B, 1, D, H, W)
        labels = batch["label"].to(device)            # (B,)

        optimizer.zero_grad()
        logits = model(voxels)                        # (B, num_classes)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def evaluate_model(model, loader, device):
    """
    Evaluate the model on a DataLoader and return accuracy.

    Args:
        model  : The neural network in eval mode.
        loader : DataLoader (train or test).
        device : torch.device.

    Returns:
        float: Accuracy as a percentage (0–100).
    """
    model.eval()
    correct = 0
    total   = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="  Evaluating", leave=False):
            voxels = batch["voxels"].to(device)
            labels = batch["label"].to(device)
            logits = model(voxels)
            predictions = logits.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total   += labels.size(0)

    return 100.0 * correct / total if total > 0 else 0.0

In [ ]:
def _save_best_checkpoint(model, val_accuracy, best_val_accuracy, checkpoint_path):
    """Save model weights if val_accuracy improved; return updated best accuracy."""
    if val_accuracy > best_val_accuracy:
        torch.save(model.state_dict(), checkpoint_path)
        return val_accuracy
    return best_val_accuracy


def _print_epoch_summary(epoch, num_epochs, mean_loss, val_accuracy, improved):
    """Print a one-line epoch summary with optional checkpoint indicator."""
    print(f"Epoch {epoch:>3}/{num_epochs}  "
          f"loss={mean_loss:.4f}  "
          f"val_acc={val_accuracy:.1f}%", end="")
    print("  ✓ checkpoint saved" if improved else "")


def train_model(model, train_loader, val_loader, optimizer, criterion,
                num_epochs, checkpoint_path):
    """
    Full training loop: runs num_epochs epochs, saves the best checkpoint.

    Returns:
        dict: {'train_losses': list[float], 'val_accuracies': list[float]}
    """
    best_val_accuracy = 0.0
    train_losses      = []
    val_accuracies    = []
    for epoch in range(1, num_epochs + 1):
        mean_loss    = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_accuracy = evaluate_model(model, val_loader, device)
        train_losses.append(mean_loss)
        val_accuracies.append(val_accuracy)
        new_best  = _save_best_checkpoint(model, val_accuracy, best_val_accuracy, checkpoint_path)
        improved  = new_best > best_val_accuracy
        _print_epoch_summary(epoch, num_epochs, mean_loss, val_accuracy, improved)
        best_val_accuracy = new_best
    print(f"\nTraining complete. Best validation accuracy: {best_val_accuracy:.1f}%")
    return {"train_losses": train_losses, "val_accuracies": val_accuracies}


#### Expected Output — Training

In demo mode you should see exactly 3 epoch lines, e.g.:
```
Epoch   1/3  loss=1.0467  val_acc=55.0%  ✓ checkpoint saved
Epoch   2/3  loss=0.38xx  val_acc=85.0%  ✓ checkpoint saved
Epoch   3/3  loss=0.40xx  val_acc=82.5%

Training complete. Best validation accuracy: 85.0%
```
First-epoch loss (1.05) is reproducible across kernel restarts.
Loss decreases epoch-over-epoch; val_acc climbs before potentially fluctuating.


In [ ]:
# Instantiate loss function and optimiser
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Run the training loop
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    num_epochs=NUM_EPOCHS,
    checkpoint_path=CHECKPOINT_PATH,
)

#### Expected Output — Training Curves

Two lines on the same plot:
- **Blue**: training loss (left axis) — should decrease across epochs.
- **Orange**: validation accuracy % (right axis) — should increase across epochs.

In [ ]:
# Plot training loss and validation accuracy on a dual-axis chart
epochs_axis = list(range(1, NUM_EPOCHS + 1))

fig, ax_loss = plt.subplots(figsize=(8, 4))
ax_acc = ax_loss.twinx()   # second y-axis sharing the same x-axis

line_loss, = ax_loss.plot(epochs_axis, history["train_losses"],
                          color="steelblue", marker="o", label="Train loss")
line_acc,  = ax_acc.plot(epochs_axis, history["val_accuracies"],
                         color="darkorange", marker="s", label="Val accuracy (%)")

ax_loss.set_xlabel("Epoch")
ax_loss.set_ylabel("Loss", color="steelblue")
ax_acc.set_ylabel("Validation Accuracy (%)", color="darkorange")
ax_loss.set_xticks(epochs_axis)

# Combine legends from both axes
fig.legend(handles=[line_loss, line_acc], loc="upper center",
           bbox_to_anchor=(0.5, 1.02), ncol=2)
fig.suptitle("Training Curves", fontsize=12, y=1.06)
plt.tight_layout()
plt.show()

### 🏋️ Exercise — Section 4

Add a `StepLR` scheduler that halves the learning rate every 5 epochs:

```python
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
```

Call `scheduler.step()` at the end of each epoch inside `train_model`.

1. How does the loss curve shape change with the scheduler active?

---
## 5. Evaluation & Prediction Results

### Concept: The Confusion Matrix

A confusion matrix is a C × C grid where entry `[i, j]` counts how many samples
with true label `i` were predicted as class `j`. The diagonal holds correct
predictions; off-diagonal entries reveal systematic errors.

- **Normalised** (row-sum = 1): shows the *rate* of confusion per true class,
  useful when classes have different sample sizes.
- **Unnormalised**: shows raw *counts*, useful for spotting absolute problem sizes.

### Concept: Per-Class Accuracy

Overall accuracy averages over all samples. Per-class accuracy reveals whether the
model is biased: a class with few samples can be ignored by the model while still
giving high overall accuracy. Plotting per-class bars immediately surfaces such
imbalances.

In [ ]:
# Load the best saved checkpoint
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f"Checkpoint not found at '{CHECKPOINT_PATH}'.\n"
        "Please run Section 4 (Training) first to generate a checkpoint."
    )

eval_model = build_resnet3d18_classifier(num_classes=NUM_CLASSES, in_channels=1)
eval_model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
eval_model = eval_model.to(device)
eval_model.eval()
print(f"Loaded checkpoint from '{CHECKPOINT_PATH}'")

#### Expected Output — Test Accuracy

In demo mode (2 classes, 3 epochs) expect **~80–90%** test accuracy.
Full mode (10 classes, 15 epochs, GPU) typically reaches **~75–85%**.


In [ ]:
# Collect all predictions and true labels from the test set
all_predictions = []
all_true_labels = []
all_voxel_grids = []  # kept for qualitative visualisation

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating on test set"):
        voxels = batch["voxels"].to(device)
        labels = batch["label"].to(device)
        logits = eval_model(voxels)
        preds  = logits.argmax(dim=1)

        all_predictions.extend(preds.cpu().tolist())
        all_true_labels.extend(labels.cpu().tolist())
        all_voxel_grids.extend(voxels.cpu().squeeze(1).numpy())  # (D,H,W) per sample

overall_accuracy = 100.0 * sum(
    p == t for p, t in zip(all_predictions, all_true_labels)
) / len(all_true_labels)

print(f"\nTest samples evaluated : {len(all_true_labels)}")
print(f"Overall test accuracy  : {overall_accuracy:.1f}%")

#### Expected Output — Confusion Matrices

Two heatmaps will be shown:
1. **Normalised** (values in [0, 1]): diagonal cells should be bright (close to 1.0).
2. **Unnormalised** (raw counts): reflects actual sample count distribution.

In [ ]:
def _annotate_heatmap_cells(ax, display_matrix, threshold, fmt):
    """Write a text label in each heatmap cell; white text on dark, black on light."""
    for row_idx in range(display_matrix.shape[0]):
        for col_idx in range(display_matrix.shape[1]):
            cell_value = display_matrix[row_idx, col_idx]
            text_color = "white" if cell_value > threshold else "black"
            formatted  = format(int(cell_value) if fmt == "d" else cell_value, fmt)
            ax.text(col_idx, row_idx, formatted,
                    ha="center", va="center", color=text_color, fontsize=8)


def plot_confusion_matrix(conf_matrix, class_names, title, normalize=False):
    """
    Render a confusion matrix as a coloured heatmap with cell annotations.

    Args:
        conf_matrix  (np.ndarray): Square integer matrix from sklearn.
        class_names  (list[str]):  Class labels for tick marks.
        title        (str):        Plot title.
        normalize    (bool):       If True, normalise rows to sum to 1.
    """
    display_matrix = conf_matrix.astype("float")
    if normalize:
        row_sums       = display_matrix.sum(axis=1, keepdims=True)
        display_matrix = np.where(row_sums != 0, display_matrix / row_sums, 0.0)
        fmt = ".2f"
    else:
        fmt = "d"
    fig, ax = plt.subplots(figsize=(max(6, len(class_names)), max(5, len(class_names))))
    im = ax.imshow(display_matrix, interpolation="nearest", cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)
    tick_marks = range(len(class_names))
    ax.set_xticks(tick_marks); ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticks(tick_marks); ax.set_yticklabels(class_names)
    _annotate_heatmap_cells(ax, display_matrix, display_matrix.max() / 2.0, fmt)
    ax.set_title(title)
    ax.set_ylabel("True label")
    ax.set_xlabel("Predicted label")
    plt.tight_layout()
    plt.show()


In [ ]:
conf_matrix = confusion_matrix(all_true_labels, all_predictions)
active_class_names = [LABEL_TO_CLASS[i] for i in sorted(set(all_true_labels))]

plot_confusion_matrix(conf_matrix, active_class_names,
                      title="Normalised Confusion Matrix", normalize=True)
plot_confusion_matrix(conf_matrix, active_class_names,
                      title="Confusion Matrix (counts)", normalize=False)


#### Expected Output — Per-Class Accuracy

A horizontal bar chart with one bar per active class.
Each bar shows accuracy in %; bars are annotated with the exact value.
The vertical dashed line marks overall accuracy.

In [ ]:
# Compute per-class accuracy from the confusion matrix diagonal
per_class_totals  = conf_matrix.sum(axis=1)          # true samples per class
per_class_correct = conf_matrix.diagonal()            # correctly predicted
per_class_acc = 100.0 * per_class_correct / np.maximum(per_class_totals, 1)

fig, ax = plt.subplots(figsize=(8, max(3, 0.5 * len(active_class_names))))
bars = ax.barh(active_class_names, per_class_acc, color="steelblue", edgecolor="white")

# Annotate each bar with its percentage value
for bar_obj, acc_value in zip(bars, per_class_acc):
    ax.text(bar_obj.get_width() + 0.5, bar_obj.get_y() + bar_obj.get_height() / 2,
            f"{acc_value:.1f}%", va="center", ha="left", fontsize=9)

ax.axvline(overall_accuracy, color="darkorange", linestyle="--",
           label=f"Overall {overall_accuracy:.1f}%")
ax.set_xlim(0, 110)
ax.set_xlabel("Accuracy (%)")
ax.set_title("Per-Class Accuracy on Test Set")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

#### Expected Output — Qualitative Prediction Examples

A grid of sample predictions:
- **Green title**: correct prediction.
- **Red title**: incorrect prediction.
Each subplot shows the 3D voxel grid scatter coloured by depth.

In [ ]:
def _select_prediction_indices(true_labels, predicted_labels, num_examples):
    """Return sample indices biased to include at least some incorrect predictions."""
    correct_idx   = [i for i, (t, p) in enumerate(zip(true_labels, predicted_labels)) if t == p]
    incorrect_idx = [i for i, (t, p) in enumerate(zip(true_labels, predicted_labels)) if t != p]
    num_incorrect = min(2, len(incorrect_idx))
    num_correct   = num_examples - num_incorrect
    return (correct_idx[:num_correct] + incorrect_idx[:num_incorrect])[:num_examples]


def _render_voxel_subplot(fig, num_rows, num_cols, plot_pos,
                          voxel_grid, true_class, pred_class):
    """Add one 3D voxel scatter subplot; title is green for correct, red for incorrect."""
    is_correct  = true_class == pred_class
    title_color = "green" if is_correct else "red"
    occupied    = np.argwhere(voxel_grid == 1.0)
    ax = fig.add_subplot(num_rows, num_cols, plot_pos, projection="3d")
    if len(occupied) > 0:
        ax.scatter(occupied[:, 0], occupied[:, 1], occupied[:, 2],
                   s=2, c=occupied[:, 2], cmap="viridis", alpha=0.6)
    ax.set_title(f"True: {true_class}\nPred: {pred_class}", color=title_color, fontsize=9)
    ax.set_xlabel("D"); ax.set_ylabel("H"); ax.set_zlabel("W")
    ax.set_box_aspect([1, 1, 1])


def show_prediction_examples(voxel_grids, true_labels, predicted_labels,
                              label_to_class, num_examples=6):
    """Display a grid of test samples with true and predicted class labels.
    Correct predictions are titled in green; incorrect ones in red."""
    selected_idx = _select_prediction_indices(true_labels, predicted_labels, num_examples)
    num_cols = min(num_examples, 3)
    num_rows = math.ceil(len(selected_idx) / num_cols)
    fig = plt.figure(figsize=(5 * num_cols, 4 * num_rows))
    fig.suptitle("Prediction Examples (green = correct, red = incorrect)",
                 fontsize=12, fontweight="bold")
    for plot_pos, sample_idx in enumerate(selected_idx, start=1):
        voxel_grid = voxel_grids[sample_idx]
        true_class = label_to_class[true_labels[sample_idx]]
        pred_class = label_to_class[predicted_labels[sample_idx]]
        _render_voxel_subplot(fig, num_rows, num_cols, plot_pos, voxel_grid, true_class, pred_class)
    plt.tight_layout()
    plt.show()


show_prediction_examples(
    voxel_grids=all_voxel_grids,
    true_labels=all_true_labels,
    predicted_labels=all_predictions,
    label_to_class=LABEL_TO_CLASS,
    num_examples=6,
)


### 🏋️ Exercise — Section 5

1. Identify the most confused class pair in the normalised confusion matrix
   (the off-diagonal cell with the highest value).
2. What geometric similarity between those two classes might explain the confusion?